<a href="https://colab.research.google.com/github/seo104/26-2-ESAA/blob/main/Project3_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/content/movies_train.csv')
test = pd.read_csv('/content/movies_test.csv')

#1. test 데이터 기본 현황

In [8]:
print("Train 데이터:", train.shape)
print("Test 데이터:", test.shape)

print("\n타깃 변수 존재 여부")
print("Train:", 'box_off_num' in train.columns)
print("Test:", 'box_off_num' in test.columns)

Train 데이터: (600, 12)
Test 데이터: (243, 11)

타깃 변수 존재 여부
Train: True
Test: False


#2. 결측치 및 중복 데이터 확인

In [9]:
missing = pd.DataFrame({
    '결측치 개수': test.isnull().sum(),
    '결측치 비율(%)': (test.isnull().mean() * 100).round(2)
})

display(missing)

print("중복 행 개수:", test.duplicated().sum())
print("중복 영화 제목 개수:", test['title'].duplicated().sum())

,결측치 개수,결측치 비율(%)
title,0,0.00
distributor,0,0.00
genre,0,0.00
release_time,0,0.00
time,0,0.00
screening_rat,0,0.00
director,0,0.00
dir_prev_bfnum,136,55.97
dir_prev_num,0,0.00
num_staff,0,0.00


중복 행 개수: 0
중복 영화 제목 개수: 0


- 전체 243개 데이터에서 중복 행은 발견되지 않음.
- dir_prev_bfnum의 결측치 136개는 모두 dir_prev_num == 0인 데이터와 일치.
- 따라서 이 결측치는 일반적인 데이터 누락이 아니라 감독의 이전 작품이 없다는 의미.

분석 결과: 공통 전처리 규칙에 따라 dir_prev_bfnum은 0으로 대체.

# 3. 장르 및 관람등급 분포
##3-1. 장르별 영화 편수

In [11]:
display(test['genre'].value_counts())

,count
genre,
드라마,78
멜로/로맨스,39
다큐멘터리,33
코미디,23
액션,21
공포,19
느와르,12
애니메이션,8
미스터리,7


- 드라마가 78편으로 가장 많고 전체의 약 32.1%를 차지함.
- 멜로/로맨스와 다큐멘터리가 그 뒤를 이음.
- SF, 뮤지컬, 서스펜스는 각각 1편뿐.

따라서 장르별 데이터 수에 차이가 크다는 점을 고려해야 함. 특히 소수 장르의 통계량은 대표성이 낮을 수 있음.

##3-2. 관람등급별 영화 편수

In [12]:
display(test['screening_rat'].value_counts())

,count
screening_rat,
청소년 관람불가,93
15세 관람가,73
12세 관람가,40
전체 관람가,37


청소년 관람불가 영화가 가장 많음. 다만 이 결과만으로 관람등급이 흥행에 영향을 미친다고 판단할 수는 없음.

#4. 개봉 시기 분석
개봉일을 연도와 월로 분리해서 확인
###연도별

In [13]:
year = pd.to_datetime(test['release_time']).dt.year

display(year.value_counts().sort_index())

,count
release_time,
2010,34
2011,37
2012,36
2013,35
2014,50
2015,51


###월별

In [14]:
month = pd.to_datetime(test['release_time']).dt.month

display(month.value_counts().sort_index())

,count
release_time,
1,17
2,15
3,22
4,24
5,15
6,27
7,13
8,18
9,22


특정 월에 영화가 집중되는 현상은 있음.

#5. 상영시간 분포 및 이상치 확인

In [15]:
print("===== 상영시간 기초 통계량 =====")
display(test['time'].describe())

print("===== 상영시간 180분 초과 영화 =====")
display(test.loc[test['time'] > 180, ['title', 'genre', 'time']])

print("Train 상영시간 범위:",
      train['time'].min(), "~", train['time'].max())

===== 상영시간 기초 통계량 =====


,time
count,243.000000
mean,109.798354
std,124.022711
min,40.000000
25%,91.000000
50%,104.000000
75%,114.500000
max,2015.000000


===== 상영시간 180분 초과 영화 =====


,title,genre,time
33,카페 느와르,드라마,198
224,슈퍼레이서 엔지,애니메이션,2015


Train 상영시간 범위: 45 ~ 180


- 최소 40분: 오하이오 삿포로
- 198분: 카페 느와르
- 2,015분: 슈퍼레이서 엔지

여기서 세 값을 모두 이상치로 판단하면 안 됨. 명백한 입력 오류인 2015분만 수정.

#6. 주요 수치형 변수 분석

In [16]:
cols = ['num_staff', 'num_actor', 'dir_prev_num', 'dir_prev_bfnum']

print("===== 주요 수치형 변수 통계량 =====")
display(test[cols].describe().T)

===== 주요 수치형 변수 통계량 =====


,count,mean,std,min,25%,50%,75%,max
num_staff,243.0,159.316872,1.629766e+02,0.0,18.0,105.0,2.820000e+02,776.0
num_actor,243.0,3.477366,2.111192e+00,0.0,2.0,3.0,4.000000e+00,16.0
dir_prev_num,243.0,0.851852,1.200321e+00,0.0,0.0,0.0,1.000000e+00,6.0
dir_prev_bfnum,107.0,891669.516044,1.217341e+06,34.0,62502.0,493120.0,1.080850e+06,6173099.5


- 제작 스태프 수(num_staff)가 0인 영화도 존재하지만, 제작 규모가 작은 영화일 가능성이 있으므로 임의로 이상치 처리하지 않음.
- 배우 수(num_actor)가 0인 영화도 존재하므로, 별도의 입력 오류 근거가 없다면 그대로 유지.
- 전작 수(dir_prev_num)가 0인 영화는 감독의 이전 작품이 없는 경우로 판단.
- 전작 관객 수(dir_prev_bfnum)의 결측치는 모두 전작 수가 0인 경우에 해당하므로, 이전 작품이 없는 것으로 보고 0으로 대체.

#7. Train과 Test 데이터 분포 비교

In [17]:
# 수치형 변수 비교
cols = ['time', 'num_staff', 'num_actor', 'dir_prev_num']

print("===== Train 데이터 =====")
display(train[cols].describe().T)

print("===== Test 데이터 =====")
display(test[cols].describe().T)

# 장르별 비율 비교
print("===== 장르별 비율 비교 =====")
display(pd.DataFrame({
    'Train(%)': train['genre'].value_counts(normalize=True) * 100,
    'Test(%)': test['genre'].value_counts(normalize=True) * 100
}).fillna(0).round(2))

# 관람등급별 비율 비교
print("===== 관람등급별 비율 비교 =====")
display(pd.DataFrame({
    'Train(%)': train['screening_rat'].value_counts(normalize=True) * 100,
    'Test(%)': test['screening_rat'].value_counts(normalize=True) * 100
}).fillna(0).round(2))

===== Train 데이터 =====


,count,mean,std,min,25%,50%,75%,max
time,600.0,100.863333,18.097528,45.0,89.0,100.0,114.0,180.0
num_staff,600.0,151.118333,165.654671,0.0,17.0,82.5,264.0,869.0
num_actor,600.0,3.706667,2.446889,0.0,2.0,3.0,4.0,25.0
dir_prev_num,600.0,0.876667,1.183409,0.0,0.0,0.0,2.0,5.0


===== Test 데이터 =====


,count,mean,std,min,25%,50%,75%,max
time,243.0,109.798354,124.022711,40.0,91.0,104.0,114.5,2015.0
num_staff,243.0,159.316872,162.976607,0.0,18.0,105.0,282.0,776.0
num_actor,243.0,3.477366,2.111192,0.0,2.0,3.0,4.0,16.0
dir_prev_num,243.0,0.851852,1.200321,0.0,0.0,0.0,1.0,6.0


===== 장르별 비율 비교 =====


,Train(%),Test(%)
genre,,
SF,2.17,0.41
공포,7.00,7.82
느와르,4.50,4.94
다큐멘터리,15.50,13.58
드라마,36.83,32.10
멜로/로맨스,13.00,16.05
뮤지컬,0.83,0.41
미스터리,2.83,2.88
서스펜스,0.33,0.41


===== 관람등급별 비율 비교 =====


,Train(%),Test(%)
screening_rat,,
청소년 관람불가,34.00,38.27
15세 관람가,33.67,30.04
12세 관람가,17.00,16.46
전체 관람가,15.33,15.23


- 상영 시간(time)의 중앙값은 Train 100분, Test 104분으로 큰 차이가 없음.
- 제작 스태프 수(num_staff)의 중앙값은 Train 82.5명, Test 105명으로 차이가 있지만, 데이터 분포를 고려해 별도의 이상치 처리 없이 유지.
- 배우 수(num_actor)와 감독의 전작 수(dir_prev_num)는 두 데이터 간 중앙값이 동일함.
- 장르와 상영 등급의 비율에 일부 차이는 있지만, 전반적으로 유사한 분포를 보이므로 별도의 분포 보정은 진행하지 않음.


#8. 감독 및 배급사 범주 확인
Test 데이터에 Train에는 등장하지 않는 감독이나 배급사가 얼마나 존재하는지 확인. 이를 통해 범주형 변수의 인코딩 방법을 결정하는 데 참고.

In [18]:
for col in ['director', 'distributor']:
    unseen = set(test[col].dropna()) - set(train[col].dropna())

    print(f"===== {col} =====")
    print("Train에 없는 고유값 개수:", len(unseen))
    print("Test 내 해당 데이터 개수:",
          test[col].isin(unseen).sum())

===== director =====
Train에 없는 고유값 개수: 142
Test 내 해당 데이터 개수: 154
===== distributor =====
Train에 없는 고유값 개수: 35
Test 내 해당 데이터 개수: 36


- Test 데이터의 감독 중 약 63.4%가 Train 데이터에 등장하지 않는 감독임.
- 감독 정보는 Train에서 학습하지 못한 새로운 범주가 많으므로, 모델 학습 변수에서 제외하는 방향으로 결정.
- 배급사는 표기 방식이 다른 동일한 배급사가 존재하므로, (주) 등의 표기를 통일하여 범주를 정리.
- Train에 없는 배급사 범주가 Test에 존재할 수 있으므로, 빈도 인코딩 등 새로운 범주에도 대응할 수 있는 방식으로 처리.

[결론]
1. 결측치 처리
    - dir_prev_bfnum의 결측치는 모두 dir_prev_num이 0인 경우이므로, 감독의 이전 작품이 없는 것으로 판단하여 0으로 대체한다.
    - 감독의 전작 여부를 구분하기 위해 dir_prev_num을 활용하여 has_prev 변수를 추가한다.
2. 이상치 및 데이터 오류 처리
    - 상영시간 중 40분인 '오하이오 삿포로'와 198분인 '카페 느와르'는 실제 상영시간일 가능성이 있으므로 유지한다.
    - 2,015분으로 기록된 '슈퍼레이서 엔지'는 명백한 입력 오류로 판단하여 해당 장르의 Train 데이터 상영시간 중앙값으로 수정한다.
    - 제작 스태프 수와 배우 수가 0인 데이터는 임의로 제거하거나 수정하지 않는다.
3. 범주형 변수 처리
    - 배급사는 (주) 등의 표기 차이를 통일하여 동일한 배급사가 서로 다른 범주로 인식되지 않도록 정리한다.
    - 장르와 관람등급은 원-핫 인코딩을 적용한다.
    - 배급사는 Train에 없는 새로운 범주가 Test에 존재할 수 있으므로, 빈도 인코딩을 적용하여 처리한다.
    - Test 데이터에 Train에 등장하지 않는 감독이 많으므로 감독 변수는 모델 학습에서 제외한다.
4. 변수 생성 및 모델 학습
    - 개봉일(release_time)에서 연도와 월을 추출하여 새로운 변수로 활용한다.
    - 상영시간, 제작 스태프 수, 배우 수, 감독의 전작 수 및 전작 관객 수 등의 수치형 변수를 활용한다.
    - 전처리 과정에서 Train과 Test의 범주형 변수 인코딩 결과가 동일한 형태가 되도록 처리한다.